# LLM Classification


In [151]:
from pathlib import Path
import pandas as pd
from IPython.display import display

resources_dir = Path("../resources")
dataset_path = resources_dir / "fake reviews dataset.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print(f"Dataset path: {dataset_path}")

Dataset path: ../resources/fake reviews dataset.csv


In [152]:
if not dataset_path.exists():
    print("Dataset not found locally. Running downloader.py...")
    !python downloader.py

    if not dataset_path.exists():
        raise FileNotFoundError(f"Downloader finished, but dataset is still missing: {dataset_path}")

    print(f"Dataset downloaded to: {dataset_path}")
else:
    print(f"Dataset already present: {dataset_path}")

Dataset already present: ../resources/fake reviews dataset.csv


In [153]:
import base

train_df, test_df = base.load_and_split_data()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (32345, 4)
Test shape: (8087, 4)


In [154]:
display(train_df.head(13))
display(test_df.head())

,category,rating,label,text_
0,Kindle_Store_5,5.0,OR,I love paranormal books and this one made me feel so much!!! I was happy and mad and sad and then thrilled and then ...
1,Books_5,4.0,CG,"Like almost all of James Patterson's books, the third is a boring, slow read."
2,Clothing_Shoes_and_Jewelry_5,1.0,CG,Im very disappointed with my purchase. The quality is just not what I was expecting.Very pretty.I bought this for my
3,Books_5,5.0,OR,Bought this book as a gift for my wife! Needless to say she loved it!
4,Movies_and_TV_5,5.0,CG,"DVDs were what I expected. The quality of the video is very good, the music is very good, the DVD's are a must have ..."
5,Movies_and_TV_5,1.0,CG,I don't know what agenda this movie is being made. I saw it for the first time last night and I am very happy I did....
6,Clothing_Shoes_and_Jewelry_5,5.0,OR,worked great for my dress to give me sleeves without dying of heat. Originally bought a large but was swimming in i...
7,Clothing_Shoes_and_Jewelry_5,5.0,CG,Love this top. Makes nursing more comfortable and the materials are thick enough to make it comfortable.
8,Clothing_Shoes_and_Jewelry_5,5.0,OR,I have had New Balance cross trainers before and am not disappointed.\nI walk 2 miles daily and they are very comfy....
9,Books_5,3.0,CG,i think nothing can top this book. It is a history book that is well worth the time. I had to read it before going...


,category,rating,label,text_
0,Toys_and_Games_5,1.0,OR,"Incredibly horrible quality and detail. Period. Seriously, it's nothing like the picture."
1,Pet_Supplies_5,2.0,CG,"""MINIMAL"" Attractactant, IF ANY... (Cats are very picky"
2,Clothing_Shoes_and_Jewelry_5,5.0,OR,"Promptly delivers, no qualms about anything else.\n\nGreat boot, I wear it all the time now.\n\nIt works for the off..."
3,Books_5,5.0,CG,What can i say another word about the book? I really enjoyed this book! I am a big fan of the Harry Bosch
4,Home_and_Kitchen_5,2.0,CG,Serious off-gassing!!! ugh! Not only do you have to clean the area around the


In [155]:
import time

import openai
import pandas as pd

from tqdm import tqdm
from sklearn.metrics import classification_report

In [156]:
! pwd

/Users/svetlana/Documents/CAS_ML/Text/Project/text_analysis/src


In [157]:
from openai import OpenAI

with open("openrouter_api_key.txt", "r") as f:
    openrouter_api_key = f.read().strip()

print("Key starts with:", openrouter_api_key[:8])
print("Key length:", len(openrouter_api_key))

openai_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
)

Key starts with: sk-or-v1
Key length: 73


In [158]:
#MODEL = "gpt-4o-mini" # or "gpt-4o"

In [159]:
MODEL = "openai/gpt-5.3-chat"

In [160]:
system_prompt = """Classify sentiment!"""

In [161]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Nice weather"}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0
)

print(response.choices[0].message.content)

Positive


In [162]:
system_prompt = """
You are an expert fake-review detector.

Your task is to classify product reviews as either:

CG = computer-generated or fake review
OR = original human-written review

Use the review text, rating, and product category to decide.

Look for signs of CG/fake reviews such as:
- generic language with little specific detail
- repetitive or unnatural wording
- contradictions inside the review
- overly vague praise or criticism
- strange grammar or sentence structure
- review text that feels incomplete or copied
- mismatch between rating and review sentiment
- unrealistic emotional tone
- lack of concrete product experience

Look for signs of OR/real reviews such as:
- specific personal experience
- natural emotional reactions
- concrete details about the product
- realistic imperfections in writing
- clear connection between rating and opinion

Return your answer in this exact format:

Label: CG or OR
Explanation: one short sentence explaining the reason.

Do not write anything else.
"""

In [163]:
test_review = """
DVDs were what I expected. The quality of the video is very good, the music is very good, the DVD's are a must have ...
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_review}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0
)

print(response.choices[0].message.content)

Label: CG  
Explanation: The review is overly generic with vague praise and no specific details about the product experience.


In [164]:
system_prompt = """
You are an expert fake-review detector.

Your task is to classify product reviews as either:

CG = computer-generated or fake review
OR = original human-written review

Use the review text, rating, and product category.

Important patterns from this dataset:

1. OR reviews often include specific personal experience.
Example:
Category: Kindle_Store_5
Rating: 5.0
Review: I love paranormal books and this one made me feel so much!!! I was happy and mad and sad and then thrilled and then ...
Label: OR
Reason: Emotional and personal reaction to reading the book.

2. CG reviews can be short, generic, or overly broad.
Example:
Category: Books_5
Rating: 4.0
Review: Like almost all of James Patterson's books, the third is a boring, slow read.
Label: CG
Reason: Generic statement with limited concrete detail.

3. CG reviews may contain unnatural or contradictory text.
Example:
Category: Clothing_Shoes_and_Jewelry_5
Rating: 1.0
Review: Im very disappointed with my purchase. The quality is just not what I was expecting.Very pretty.I bought this for my
Label: CG
Reason: The text is incomplete and contradictory.

4. OR reviews may be simple but still natural.
Example:
Category: Books_5
Rating: 5.0
Review: Bought this book as a gift for my wife! Needless to say she loved it!
Label: OR
Reason: Short but believable personal purchasing experience.

5. CG reviews may sound repetitive and promotional.
Example:
Category: Movies_and_TV_5
Rating: 5.0
Review: DVDs were what I expected. The quality of the video is very good, the music is very good, the DVD's are a must have ...
Label: CG
Reason: Repetitive praise and generic wording.

6. CG reviews may have sentiment/rating inconsistency.
Example:
Category: Movies_and_TV_5
Rating: 1.0
Review: I don't know what agenda this movie is being made. I saw it for the first time last night and I am very happy I did....
Label: CG
Reason: Negative rating conflicts with positive wording.

7. OR reviews often mention concrete usage details.
Example:
Category: Clothing_Shoes_and_Jewelry_5
Rating: 5.0
Review: worked great for my dress to give me sleeves without dying of heat. Originally bought a large but was swimming in i...
Label: OR
Reason: Specific use case and sizing detail.

8. CG reviews may sound smooth but generic.
Example:
Category: Clothing_Shoes_and_Jewelry_5
Rating: 5.0
Review: Love this top. Makes nursing more comfortable and the materials are thick enough to make it comfortable.
Label: CG
Reason: General praise with limited personal detail.

9. OR reviews often include real-life habits or repeated use.
Example:
Category: Clothing_Shoes_and_Jewelry_5
Rating: 5.0
Review: I have had New Balance cross trainers before and am not disappointed.
I walk 2 miles daily and they are very comfy....
Label: OR
Reason: Specific brand history and daily walking use.

10. CG reviews may contain vague praise that does not fit the rating well.
Example:
Category: Books_5
Rating: 3.0
Review: i think nothing can top this book. It is a history book that is well worth the time. I had to read it before going...
Label: CG
Reason: Strong praise does not match the medium rating.

11. CG reviews may repeat generic positive phrases without giving specific evidence.
Example:
Category: Books_5
Rating: 4.0
Review: I do like this book, and it's an interesting and good read. I would recommend it.I read this book in the middle of ...
Label: CG
Reason: The review uses broad praise like interesting, good read, and recommend it, but gives little concrete detail.

12. OR reviews can be very short when they contain a specific personal habit or use case.
Example:
Category: Electronics_5
Rating: 5.0
Review: Love my p-Touch - i label everything that doesn't move.
Label: OR
Reason: Short but natural, with a specific personal use habit.

Classification rules:
- Predict CG when the review is generic, repetitive, contradictory, incomplete, or inconsistent with the rating.
- Predict OR when the review contains a believable personal experience, concrete use case, specific product detail, or natural emotional reaction.
- Do not assume a review is OR only because it has grammar mistakes.
- Do not assume a review is CG only because it is short.
- Focus on the style and consistency of the review.

Return only valid JSON in this exact format:

{
  "label": "CG or OR"
}

Do not write markdown.
Do not write anything outside the JSON.
"""

In [165]:
def classify_review_json(category, rating, review_text):
    user_prompt = f"""
Category: {category}
Rating: {rating}
Review: {review_text}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0
    )

    content = response.choices[0].message.content.strip()
    return json.loads(content)

In [166]:
from tqdm.notebook import tqdm
tqdm.pandas()

In [167]:
test_50 = test_df.head(50).copy()

results = test_50.apply(
    lambda row: classify_review_json(
        category=row["category"],
        rating=row["rating"],
        review_text=row["text_"]
    ),
    axis=1
)

test_50["predicted_label"] = results.apply(lambda x: x["label"])
test_50["explanation"] = results.apply(lambda x: x.get("explanation", ""))

test_50[["category", "rating", "label", "predicted_label", "explanation", "text_"]]

,category,rating,label,predicted_label,explanation,text_
0,Toys_and_Games_5,1.0,OR,OR,,"Incredibly horrible quality and detail. Period. Seriously, it's nothing like the picture."
1,Pet_Supplies_5,2.0,CG,CG,,"""MINIMAL"" Attractactant, IF ANY... (Cats are very picky"
2,Clothing_Shoes_and_Jewelry_5,5.0,OR,OR,,"Promptly delivers, no qualms about anything else.\n\nGreat boot, I wear it all the time now.\n\nIt works for the off..."
3,Books_5,5.0,CG,CG,,What can i say another word about the book? I really enjoyed this book! I am a big fan of the Harry Bosch
4,Home_and_Kitchen_5,2.0,CG,CG,,Serious off-gassing!!! ugh! Not only do you have to clean the area around the
5,Books_5,5.0,CG,CG,,A mind is a terrible thing. The only way you can be a good person is to live.\n\nI had a friend who was a social wor...
6,Tools_and_Home_Improvement_5,4.0,CG,OR,,easy to carry and hole saw. The only problem is that it comes with a small screw driver.
7,Books_5,5.0,OR,OR,,"Just received today--another great one from Ina. My wife has many of these books. Each has go-to recipes, and this i..."
8,Electronics_5,4.0,OR,OR,,Really fun kit to get you familiar with a variety of light modifiers.\n\nHonestly more durable than expected which c...
9,Pet_Supplies_5,5.0,OR,CG,,Great product. Love the wheels too. And PBA free. Important for our babies too.


In [168]:
# Import metrics from scikit-learn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [169]:
# True labels from your dataset
# These are the real labels: CG or OR
y_true = test_50["label"]

# Predicted labels from ChatGPT
# These are the labels generated by your prompt/model
y_pred = test_50["predicted_label"]

In [170]:
# Print a full classification report
# This shows precision, recall, and F1 score for both CG and OR
print(classification_report(y_true, y_pred, labels=["CG", "OR"]))

              precision    recall  f1-score   support

          CG       0.89      0.85      0.87        20
          OR       0.90      0.93      0.92        30

    accuracy                           0.90        50
   macro avg       0.90      0.89      0.89        50
weighted avg       0.90      0.90      0.90        50



In [171]:
# Create a confusion matrix
# It shows where the model was correct and where it made mistakes
cm = confusion_matrix(y_true, y_pred, labels=["CG", "OR"])

# Convert confusion matrix into a readable table
cm_df = pd.DataFrame(
    cm,
    index=["True CG", "True OR"],
    columns=["Predicted CG", "Predicted OR"]
)

# Display the confusion matrix table
cm_df

,Predicted CG,Predicted OR
True CG,17,3
True OR,2,28
